In [1]:
from incidentiq.search import SearchEngine
from incidentiq.context.builder import ContextBuilder
from incidentiq.context.patterns import extract_patterns
from incidentiq.reasoning.prompt import build_reasoning_prompt

e:\incidentiq\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_PATH = "../data/processed/logs.parquet"

engine = SearchEngine(DATA_PATH)
context_builder = ContextBuilder(engine.df)

Batches: 100%|██████████| 63/63 [00:04<00:00, 14.38it/s]


In [3]:
query = "node card failure"

results = engine.search_hybrid(query, top_k=10)

results

[{'rank': 1,
  'doc_id': 457,
  'log_id': np.int64(458),
  'timestamp': Timestamp('2005-06-28 09:53:39.479164'),
  'node': 'R02-M1-NE',
  'severity': 'WARNING',
  'message': 'Node card is not fully functional',
  'score': 0.032266458495966696},
 {'rank': 2,
  'doc_id': 1948,
  'log_id': np.int64(1949),
  'timestamp': Timestamp('2005-12-06 10:05:04.300635'),
  'node': 'R12-M0-NC',
  'severity': 'WARNING',
  'message': 'Node card is not fully functional',
  'score': 0.03177805800756621},
 {'rank': 3,
  'doc_id': 1229,
  'log_id': np.int64(1230),
  'timestamp': Timestamp('2005-08-09 10:53:17.485009'),
  'node': 'R74-M0-N1',
  'severity': 'WARNING',
  'message': 'Node card is not fully functional',
  'score': 0.0315136476426799},
 {'rank': 4,
  'doc_id': 1227,
  'log_id': np.int64(1228),
  'timestamp': Timestamp('2005-08-09 10:40:46.749252'),
  'node': 'R51-M1-ND',
  'severity': 'WARNING',
  'message': 'Node card is not fully functional',
  'score': 0.03149801587301587},
 {'rank': 5,
  'do

In [4]:
context = context_builder.build_context(results)
context.keys()

dict_keys(['evidence', 'groups', 'timeline', 'statistics', 'time_range', 'temporal_clusters'])

In [5]:
patterns = extract_patterns(context)
patterns

[{'type': 'repeated_message',
  'message': 'Node card is not fully functional',
  'occurrences': 6,
  'evidence_ids': [457, 1948, 1229, 1227, 1218, 620]},
 {'type': 'repeated_message',
  'message': 'Can not get assembly information for node card',
  'occurrences': 2,
  'evidence_ids': [522, 1204]},
 {'type': 'affected_nodes', 'count': 10},
 {'type': 'severity_distribution',
  'distribution': {'WARNING': 6, 'INFO': 2, 'SEVERE': 2}},
 {'type': 'temporal_cluster',
  'event_count': 2,
  'start': Timestamp('2005-08-09 10:40:46.749252'),
  'end': Timestamp('2005-08-09 10:53:17.485009'),
  'duration': Timedelta('0 days 00:12:30.735757')}]

In [6]:
prompt = build_reasoning_prompt(
    query=query,
    context=context,
    patterns=patterns,
)
print(prompt)

INCIDENT QUERY
node card failure

EVIDENCE
[457] 2005-06-28 09:53:39.479164 | R02-M1-NE | WARNING | Node card is not fully functional | 
[1948] 2005-12-06 10:05:04.300635 | R12-M0-NC | WARNING | Node card is not fully functional | 
[1229] 2005-08-09 10:53:17.485009 | R74-M0-N1 | WARNING | Node card is not fully functional | 
[1227] 2005-08-09 10:40:46.749252 | R51-M1-ND | WARNING | Node card is not fully functional | 
[1218] 2005-08-04 10:58:22.010246 | R06-M1-ND | WARNING | Node card is not fully functional | 
[620] 2005-07-08 23:15:03.798449 | R05-M0-N2 | WARNING | Node card is not fully functional | 
[1801] 2005-11-29 16:21:56.855025 | R67-M1-N7 | INFO | Node card VPD check: U01 node in processor card slot J05 do not match. VPD ecid 04D37DF2DE7BFFFF0D081AF0DAD2, found 04DD80740E2FFFFF0A0C19D0CEBD | 
[522] 2005-07-01 11:05:31.120732 | R37-M1-N4 | SEVERE | Can not get assembly information for node card | 
[1413] 2005-09-20 11:57:30.636832 | R05-M0-NA | INFO | Node card VPD check: U01 

In [7]:
from incidentiq.reasoning.analyzer import IncidentAnalyzer

In [8]:
analyzer = IncidentAnalyzer()

In [9]:
from incidentiq.reasoning.models import IncidentAnalysis

In [10]:
analysis = analyzer.analyze(
    query=query,
    context=context,
    patterns=patterns,
)

NotFoundError: Error code: 404 - {'error': {'message': "Model 'gemini-3.6.-flash' not found. Did you mean 'gemini-3.6-flash'? Please verify the model name against the supported list: https://ai.google.dev/gemini-api/docs/models", 'code': 'not_found'}}

In [ ]:
analysis

IncidentAnalysis(summary='Multiple node card issues were recorded between June 28, 2005, and December 6, 2005. Events include warnings about non-fully functional node cards, severe errors regarding inability to retrieve assembly information, and informational logs regarding Vital Product Data (VPD) check mismatches.', observations=[Observation(statement='Six warning logs indicate that node cards were not fully functional across locations R02-M1-NE, R05-M0-N2, R06-M1-ND, R51-M1-ND, R74-M0-N1, and R12-M0-NC.', evidence_ids=[457, 620, 1218, 1227, 1229, 1948]), Observation(statement='Two severe logs indicate that assembly information for node cards could not be retrieved at location R37-M1-N4 and UNKNOWN_LOCATION.', evidence_ids=[522, 1204]), Observation(statement='Two informational logs report Node card VPD check failures due to ecid mismatches in processor card slots J05 (R67-M1-N7) and J15 (R05-M0-NA).', evidence_ids=[1413, 1801])], hypotheses=[], unknowns=['The physical location associ